In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import numpy as np

In [ ]:
# 1.Analyze the score of Llama prompts
# Load the dataset
df = pd.read_csv('Rating for llama prompts.csv')

# Define the scaling factors as requested
# Manual and G-Eval are scaled by 100
# GPT and Llama are scaled by 20
df['manual_scaled'] = (df['Manual rate'].fillna(0) * 100).astype(int)
df['geval_scaled'] = (df['G-Eval rate'].fillna(0) * 100).astype(int)
df['gpt_scaled'] = (df['GPT rate'].fillna(0) * 20).astype(int)
df['llama_scaled'] = (df['Llama rate'].fillna(0) * 20).astype(int)

print("Data Normalization Complete.")
print(df[['manual_scaled', 'geval_scaled', 'gpt_scaled', 'llama_scaled']].head())

In [ ]:
def calculate_kappa(rater_a, rater_b, name):
    # We use quadratic weights because these are numerical rankings
    simple_kappa = cohen_kappa_score(rater_a, rater_b)
    weighted_kappa = cohen_kappa_score(rater_a, rater_b, weights='quadratic')

    print(f"--- Comparison: Manual Rate vs {name} ---")
    print(f"Simple Kappa:    {simple_kappa:.4f}")
    print(f"Weighted Kappa:  {weighted_kappa:.4f}\n")
    return weighted_kappa

# List of AI raters to compare against Manual Rate
ai_raters = {
    'G-Eval Rate': df['geval_scaled'],
    'GPT Rate': df['gpt_scaled'],
    'Llama Rate': df['llama_scaled']
}

results = {}
for name, data in ai_raters.items():
    results[name] = calculate_kappa(df['manual_scaled'], data, name)

In [ ]:
# 2.Analyze the score of GPT prompt
# Load the dataset
df = pd.read_csv('Rating for gpt prompts.csv')

# 1. Scale Manual and AI Model rates by 20 (assuming 1-5 scale)
df['manual_scaled'] = (df['Manual'].fillna(0) * 20).astype(int)
df['gpt_scaled'] = (df['gpt_rate'].fillna(0) * 20).astype(int)
df['claude_scaled'] = (df['claude_rate'].fillna(0) * 20).astype(int)
df['llama_scaled'] = (df['llama_rate'].fillna(0) * 20).astype(int)

# 2. Scale the last two G-Eval columns by 100 (assuming 0-1 scale)
df['geval_gpt_scaled'] = (df['G-Eval with gpt rate'].fillna(0) * 100).astype(int)
df['geval_claude_scaled'] = (df['G-Eval with Claude rate'].fillna(0) * 100).astype(int)

print("Normalization Complete. All scores scaled to 100.")
print(df[['manual_scaled', 'gpt_scaled', 'claude_scaled', 'llama_scaled', 'geval_gpt_scaled', 'geval_claude_scaled']].head())

In [ ]:
def calculate_kappa(rater_a, rater_b, name):
    # Standard and Quadratic Weighted Kappa
    simple_kappa = cohen_kappa_score(rater_a, rater_b)
    weighted_kappa = cohen_kappa_score(rater_a, rater_b, weights='quadratic')

    print(f"--- Manual vs {name} ---")
    print(f"Simple Kappa:    {simple_kappa:.4f}")
    print(f"Weighted Kappa:  {weighted_kappa:.4f}\n")
    return weighted_kappa

# Dictionary of columns to compare against Manual
comparisons = {
    'GPT Rate': df['gpt_scaled'],
    'Claude Rate': df['claude_scaled'],
    'Llama Rate': df['llama_scaled'],
    'G-Eval (GPT)': df['geval_gpt_scaled'],
    'G-Eval (Claude)': df['geval_claude_scaled']
}

results = {}
for label, data in comparisons.items():
    results[label] = calculate_kappa(df['manual_scaled'], data, label)

In [ ]:
# Create a summary table for easy viewing
summary_df = pd.DataFrame.from_dict(results, orient='index', columns=['Weighted Kappa'])
summary_df = summary_df.sort_values(by='Weighted Kappa', ascending=False)

print("Summary Ranking (Agreement with Manual):")
print(summary_df)

In [ ]:
# 3.Analyze the score of G-Eval with GPT as judge
df = pd.read_csv('Rating for mixtral prompt.csv')

# Scaling: Manual, Llama, and GPT rates are * 20
df['manual_scaled'] = (df['Manual rate'].fillna(0) * 20).astype(int)
df['llama_scaled'] = (df['llama_rate'].fillna(0) * 20).astype(int)
df['gpt_scaled'] = (df['GPT rate'].fillna(0) * 20).astype(int)

# Scaling: G-Eval is * 100
df['geval_scaled'] = (df['G-Eval Rate'].fillna(0) * 100).astype(int)

print("Normalization Complete (Target Scale: 100).")
print(df[['manual_scaled', 'llama_scaled', 'gpt_scaled', 'geval_scaled']].head())

In [ ]:
def calculate_kappa(rater_a, rater_b, name):
    # Weighted Kappa is used to account for the magnitude of disagreement in scores
    simple_kappa = cohen_kappa_score(rater_a, rater_b)
    weighted_kappa = cohen_kappa_score(rater_a, rater_b, weights='quadratic')

    print(f"--- Comparison: Manual Rate vs {name} ---")
    print(f"Simple Kappa:    {simple_kappa:.4f}")
    print(f"Weighted Kappa:  {weighted_kappa:.4f}\n")
    return weighted_kappa

# List of evaluators to compare
comparisons = {
    'Llama Rate': df['llama_scaled'],
    'GPT Rate': df['gpt_scaled'],
    'G-Eval Rate': df['geval_scaled']
}

results = {}
for label, data in comparisons.items():
    results[label] = calculate_kappa(df['manual_scaled'], data, label)

In [ ]:
# Create a summary table to compare which model aligns best with human judgment
summary_df = pd.DataFrame.from_dict(results, orient='index', columns=['Quadratic Weighted Kappa'])
summary_df = summary_df.sort_values(by='Quadratic Weighted Kappa', ascending=False)

print("Final Summary Table (Agreement with Human Manual Rate):")
print("-" * 55)
print(summary_df)
print("-" * 55)